# Combined Statistical Analysis — Revision V2

This notebook performs the corrected cross-model statistical analysis for the three disease-specific experiments.

## Required files
Place these three files in the same working directory:

- `HD_V2_repeated_CV_scores.csv` (or rename the heart-disease repeated-CV output accordingly)
- `LC_V2_repeated_CV_scores.csv`
- `DB_V2_repeated_CV_scores.csv`

Each file must contain one row per model per repeated-CV split and the columns:
`Model`, `Split`, `Accuracy`, `Balanced_Accuracy`, `Precision`, `Sensitivity`, `F1`, `ROC_AUC`, `PR_AUC`.

### Statistical design
The 25 folds from 5-fold × 5-repeat CV are **not treated as 25 independent observations**. Instead, the five folds within each repeat are averaged, producing one value per model for each repeat. Across three datasets this yields 15 matched dataset–repeat blocks.

The primary endpoint is **Balanced Accuracy**. ROC–AUC and F1-score are secondary endpoints. For each endpoint:
1. Friedman test compares the five algorithms across matched dataset–repeat blocks.
2. If the omnibus test is significant, pairwise Wilcoxon signed-rank tests are performed.
3. Holm correction controls family-wise error across the 10 pairwise comparisons.
4. Rank-biserial correlation is reported as an effect-size descriptor.

This structure avoids the invalid practice of treating different metrics as independent observations.


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from scipy.stats import friedmanchisquare, wilcoxon

FILES = {
    "Heart Disease": "HD_V2_repeated_CV_scores.csv",
    "Lung Cancer": "LC_V2_repeated_CV_scores.csv",
    "Diabetes": "DB_V2_repeated_CV_scores.csv",
}

frames=[]
for dataset, fn in FILES.items():
    p=Path(fn)
    if not p.exists():
        raise FileNotFoundError(
            f"{fn} was not found. Copy the repeated-CV CSV generated by the corresponding V2 notebook "
            f"into this notebook's working directory and use the filename shown above."
        )
    d=pd.read_csv(p)
    d["Dataset"]=dataset
    frames.append(d)

cv=pd.concat(frames,ignore_index=True)
required={"Model","Split","Accuracy","Balanced_Accuracy","Precision","Sensitivity","F1","ROC_AUC","PR_AUC","Dataset"}
missing=required-set(cv.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Split numbering in the V2 notebooks is 1..25; every consecutive five folds are one repeat.
cv["Repeat"]=((cv["Split"].astype(int)-1)//5)+1
print(cv.groupby(["Dataset","Model"]).size())
display(cv.head())


Dataset        Model           
Diabetes       AdaBoost            25
               CatBoost            25
               GradientBoosting    25
               LightGBM            25
               XGBoost             25
Heart Disease  AdaBoost            25
               CatBoost            25
               GradientBoosting    25
               LightGBM            25
               XGBoost             25
Lung Cancer    AdaBoost            25
               CatBoost            25
               GradientBoosting    25
               LightGBM            25
               XGBoost             25
dtype: int64


,Model,Split,Accuracy,Balanced_Accuracy,Precision,Sensitivity,F1,ROC_AUC,PR_AUC,Dataset,Repeat
0,XGBoost,1,0.710914,0.650249,0.277512,0.563107,0.371795,0.707303,0.347441,Heart Disease,1
1,XGBoost,2,0.668142,0.629016,0.245833,0.572816,0.344023,0.679004,0.276265,Heart Disease,1
2,XGBoost,3,0.706490,0.671549,0.285714,0.621359,0.391437,0.713482,0.316841,Heart Disease,1
3,XGBoost,4,0.672566,0.637346,0.254167,0.586538,0.354651,0.704268,0.326371,Heart Disease,1
4,XGBoost,5,0.648968,0.650965,0.251852,0.653846,0.363636,0.706748,0.312444,Heart Disease,1


## Aggregate folds within each repeat

In [3]:
metrics=["Accuracy","Balanced_Accuracy","Precision","Sensitivity","F1","ROC_AUC","PR_AUC"]
repeat_mean=(cv.groupby(["Dataset","Repeat","Model"],as_index=False)[metrics].mean())
display(repeat_mean.head(15))
repeat_mean.to_csv("Combined_dataset_repeat_scores.csv",index=False)


,Dataset,Repeat,Model,Accuracy,Balanced_Accuracy,Precision,Sensitivity,F1,ROC_AUC,PR_AUC
0,Diabetes,1,AdaBoost,0.762991,0.769493,0.628613,0.790985,0.699562,0.836825,0.722709
1,Diabetes,1,CatBoost,0.766836,0.775083,0.635170,0.802166,0.707100,0.841123,0.723169
2,Diabetes,1,GradientBoosting,0.775944,0.780345,0.649376,0.794689,0.712822,0.839120,0.730035
3,Diabetes,1,LightGBM,0.746015,0.753902,0.608118,0.779804,0.682395,0.826266,0.704568
4,Diabetes,1,XGBoost,0.759036,0.765571,0.624440,0.787142,0.695655,0.837679,0.727140
5,Diabetes,2,AdaBoost,0.765623,0.769789,0.634686,0.783578,0.700715,0.842069,0.736885
6,Diabetes,2,CatBoost,0.776063,0.781336,0.650324,0.798672,0.715122,0.845268,0.736954
7,Diabetes,2,GradientBoosting,0.777328,0.777937,0.656072,0.779874,0.711593,0.842164,0.726556
8,Diabetes,2,LightGBM,0.743519,0.745157,0.613036,0.750314,0.672852,0.835100,0.718572
9,Diabetes,2,XGBoost,0.753900,0.755573,0.622502,0.761146,0.684394,0.840143,0.726760


## Descriptive repeated-CV summary

In [4]:
summary=(repeat_mean.groupby(["Dataset","Model"])[metrics].agg(["mean","std"]))
display(summary)
summary.to_csv("Combined_repeatedCV_descriptive_summary.csv")


Accuracy           Balanced_Accuracy  \
                                    mean       std              mean   
Dataset       Model                                                    
Diabetes      AdaBoost          0.756226  0.013456          0.761135   
              CatBoost          0.766120  0.008957          0.771736   
              GradientBoosting  0.772608  0.007034          0.774608   
              LightGBM          0.751797  0.009792          0.756066   
              XGBoost           0.752824  0.011590          0.757322   
Heart Disease AdaBoost          0.661888  0.003860          0.650466   
              CatBoost          0.682478  0.005061          0.645337   
              GradientBoosting  0.673392  0.003848          0.644722   
              LightGBM          0.673982  0.003663          0.635865   
              XGBoost           0.677522  0.003159          0.640504   
Lung Cancer   AdaBoost          0.848525  0.014431          0.816429   
              CatBoost          0.897726  0.008750          0.761865   
              GradientBoosting  0.886071  0.009063          0.842169   
              LightGBM          0.896457  0.010257          0.771772   
              XGBoost           0.891941  0.017770          0.773439   

                                         Precision           Sensitivity  \
                                     std      mean       std        mean   
Dataset       Model                                                        
Diabetes      AdaBoost          0.012956  0.622439  0.017906    0.777470   
              CatBoost          0.008509  0.635203  0.013437    0.790273   
              GradientBoosting  0.005802  0.646862  0.012495    0.781216   
              LightGBM          0.010352  0.618004  0.011630    0.770133   
              XGBoost           0.011184  0.619103  0.016654    0.772243   
Heart Disease AdaBoost          0.008535  0.255351  0.006041    0.634040   
              CatBoost          0.005118  0.261567  0.004882    0.591901   
              GradientBoosting  0.002132  0.257213  0.002130    0.603480   
              LightGBM          0.006824  0.252686  0.004443    0.581031   
              XGBoost           0.004607  0.256699  0.004101    0.587248   
Lung Cancer   AdaBoost          0.024698  0.963100  0.006873    0.860000   
              CatBoost          0.031850  0.939947  0.008698    0.944444   
              GradientBoosting  0.011317  0.966264  0.003796    0.901481   
              LightGBM          0.015517  0.942853  0.004890    0.939259   
              XGBoost           0.028340  0.943733  0.007079    0.932593   

                                                F1             ROC_AUC  \
                                     std      mean       std      mean   
Dataset       Model                                                      
Diabetes      AdaBoost          0.012618  0.690511  0.015199  0.835375   
              CatBoost          0.011701  0.703065  0.010311  0.842984   
              GradientBoosting  0.010137  0.706333  0.007530  0.840095   
              LightGBM          0.015187  0.684556  0.011395  0.832301   
              XGBoost           0.013219  0.686191  0.013279  0.835606   
Heart Disease AdaBoost          0.015859  0.363856  0.008622  0.707650   
              CatBoost          0.009232  0.362610  0.005810  0.699898   
              GradientBoosting  0.006466  0.360483  0.002054  0.701259   
              LightGBM          0.015924  0.352013  0.006825  0.694684   
              XGBoost           0.008453  0.357024  0.005153  0.699047   
Lung Cancer   AdaBoost          0.012669  0.907572  0.009401  0.903175   
              CatBoost          0.007407  0.941544  0.005076  0.891283   
              GradientBoosting  0.011887  0.931947  0.006134  0.931230   
              LightGBM          0.014953  0.940346  0.006690  0.884471   
              XGBoost           0.016439  0.937528  0.010718  0.899683   

                                 

## Friedman omnibus tests

In [5]:
models=sorted(repeat_mean["Model"].unique())
endpoints=["Balanced_Accuracy","ROC_AUC","F1"]

friedman_rows=[]
for metric in endpoints:
    pvt=repeat_mean.pivot_table(index=["Dataset","Repeat"],columns="Model",values=metric)
    stat,p=friedmanchisquare(*[pvt[m].values for m in models])
    friedman_rows.append({"Metric":metric,"Friedman_chi2":stat,"df":len(models)-1,"p_value":p})
friedman_df=pd.DataFrame(friedman_rows)
display(friedman_df)
friedman_df.to_csv("Combined_Friedman_tests.csv",index=False)


,Metric,Friedman_chi2,df,p_value
0,Balanced_Accuracy,26.293333,4,0.000028
1,ROC_AUC,21.173333,4,0.000293
2,F1,15.093333,4,0.004512


## Pairwise Wilcoxon signed-rank tests with Holm correction

In [6]:
def holm_adjust(pvals):
    pvals=np.asarray(pvals,float)
    order=np.argsort(pvals)
    adjusted=np.empty_like(pvals)
    running=0.0
    m=len(pvals)
    for rank, idx in enumerate(order):
        val=(m-rank)*pvals[idx]
        running=max(running,val)
        adjusted[idx]=min(running,1.0)
    return adjusted

def rank_biserial_from_diffs(d):
    d=np.asarray(d,float)
    d=d[d!=0]
    if len(d)==0:
        return 0.0
    absd=np.abs(d)
    ranks=pd.Series(absd).rank(method="average").to_numpy()
    wpos=ranks[d>0].sum()
    wneg=ranks[d<0].sum()
    denom=wpos+wneg
    return (wpos-wneg)/denom if denom else 0.0

pairwise_all=[]
for metric in endpoints:
    pvt=repeat_mean.pivot_table(index=["Dataset","Repeat"],columns="Model",values=metric)
    tmp=[]
    for a,b in combinations(models,2):
        x=pvt[a].values
        y=pvt[b].values
        diff=x-y
        try:
            stat,p=wilcoxon(x,y,zero_method="wilcox",alternative="two-sided",method="auto")
        except ValueError:
            stat,p=np.nan,1.0
        tmp.append({
            "Metric":metric,
            "Model_A":a,
            "Model_B":b,
            "Mean_A":x.mean(),
            "Mean_B":y.mean(),
            "Median_Difference_A_minus_B":np.median(diff),
            "Wilcoxon_W":stat,
            "Raw_p":p,
            "Rank_Biserial_r":rank_biserial_from_diffs(diff)
        })
    adj=holm_adjust([r["Raw_p"] for r in tmp])
    for r,padj in zip(tmp,adj):
        r["Holm_Adjusted_p"]=padj
        r["Significant_0.05"]=padj<0.05
    pairwise_all.extend(tmp)

pairwise_df=pd.DataFrame(pairwise_all)
display(pairwise_df)
pairwise_df.to_csv("Combined_pairwise_Wilcoxon_Holm.csv",index=False)


,Metric,Model_A,Model_B,Mean_A,Mean_B,Median_Difference_A_minus_B,Wilcoxon_W,Raw_p,Rank_Biserial_r,Holm_Adjusted_p,Significant_0.05
0,Balanced_Accuracy,AdaBoost,CatBoost,0.742677,0.726313,0.003965,46.0,0.454285,0.233333,0.908386,False
1,Balanced_Accuracy,AdaBoost,GradientBoosting,0.742677,0.753833,-0.009028,29.0,0.083252,-0.516667,0.416260,False
2,Balanced_Accuracy,AdaBoost,LightGBM,0.742677,0.721235,0.018890,12.0,0.004272,0.800000,0.034180,True
3,Balanced_Accuracy,AdaBoost,XGBoost,0.742677,0.723755,0.013784,19.0,0.018066,0.683333,0.108398,False
4,Balanced_Accuracy,CatBoost,GradientBoosting,0.726313,0.753833,-0.005007,18.0,0.015076,-0.700000,0.105530,False
5,Balanced_Accuracy,CatBoost,LightGBM,0.726313,0.721235,0.011347,31.0,0.106995,0.483333,0.427979,False
6,Balanced_Accuracy,CatBoost,XGBoost,0.726313,0.723755,0.004922,41.0,0.302795,0.316667,0.908386,False
7,Balanced_Accuracy,GradientBoosting,LightGBM,0.753833,0.721235,0.019820,0.0,0.000061,1.000000,0.000610,True
8,Balanced_Accuracy,GradientBoosting,XGBoost,0.753833,0.723755,0.020512,1.0,0.000122,0.983333,0.001099,True
9,Balanced_Accuracy,LightGBM,XGBoost,0.721235,0.723755,-0.002079,45.0,0.421204,-0.250000,0.908386,False


## Average model ranks across the 15 matched dataset–repeat blocks

In [7]:
rank_frames=[]
for metric in endpoints:
    r=repeat_mean.copy()
    r["Rank"]=r.groupby(["Dataset","Repeat"])[metric].rank(method="average",ascending=False)
    rr=r.groupby("Model")["Rank"].mean().rename(metric+"_Mean_Rank")
    rank_frames.append(rr)

rank_table=pd.concat(rank_frames,axis=1)
rank_table["Overall_Mean_Rank"]=rank_table.mean(axis=1)
rank_table=rank_table.sort_values("Overall_Mean_Rank")
display(rank_table)
rank_table.to_csv("Combined_model_average_ranks.csv")


,Balanced_Accuracy_Mean_Rank,ROC_AUC_Mean_Rank,F1_Mean_Rank,Overall_Mean_Rank
Model,,,,
GradientBoosting,1.666667,1.800000,2.466667,1.977778
CatBoost,2.733333,2.800000,1.866667,2.466667
AdaBoost,2.533333,2.666667,3.533333,2.911111
XGBoost,3.733333,3.400000,3.466667,3.533333
LightGBM,4.333333,4.333333,3.666667,4.111111


## Interpretation rules for the manuscript

- A smaller mean rank indicates better relative performance.
- Report the Friedman test first.
- Only emphasize pairwise Wilcoxon comparisons when the corresponding omnibus Friedman test is significant.
- Use Holm-adjusted p-values rather than raw pairwise p-values.
- Do not claim one model is universally superior if significant differences are absent.
- Because repeated cross-validation observations are correlated, interpret inferential p-values as supportive evidence rather than as a substitute for independent external validation.
